## config

In [17]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MultiLabelBinarizer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.feature_extraction.text import CountVectorizer
import matplotlib.pyplot as plt
import re

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import PCA

import json
import glob

from googleapiclient.discovery import build

In [18]:
df = pd.read_json("../../data/processed/ve_channels/ve_with_features.json")
display(df.head())
display(df.info())

,channel_id,channel_name,title,description,country,topics,keywords,uploads,videos
0,UCqgh7QNlVV8kv3LBTJZYWrg,The Film Essay,The Film Essay,A video essay channel devoted to making videos...,GB,[https://en.wikipedia.org/wiki/Television_prog...,"""film essay"" ""video essay"" ""film analysis"" ""fi...",UUqgh7QNlVV8kv3LBTJZYWrg,[{'title': 'Why Christopher Nolan uses time as...
1,UCjFqcJQXGZ6T6sxyFB-5i6A,Every Frame a Painting,Every Frame a Painting,Taylor & Tony are gone again! Thanks for watch...,None,[https://en.wikipedia.org/wiki/Lifestyle_(soci...,"filmmaking ""film analysis"" ""video essay""",UUjFqcJQXGZ6T6sxyFB-5i6A,[{'title': 'Every Frame x Lackadaisy - Intervi...
2,UCr8r7UVsDaRiIQSVUHVBd_A,Nikki Carreon,Nikki Carreon,"Hello!! Just here making silly stuff, making s...",US,[https://en.wikipedia.org/wiki/Lifestyle_(soci...,None,UUr8r7UVsDaRiIQSVUHVBd_A,"[{'title': 'Why I Went to a MENTAL HOSPITAL.',..."
3,UCjKSoJlPgcK6BmoSqXpj5xQ,Action Button,Action Button,patreon.com/actionbutton\ntwitch.tv/actionbutt...,None,[https://en.wikipedia.org/wiki/Role-playing_vi...,None,UUjKSoJlPgcK6BmoSqXpj5xQ,"[{'title': 'ACTION BUTTON PICTURES PRESENTS ""L..."
4,UCUyvQV2JsICeLZP4c_h40kA,Thomas Flight,Thomas Flight,"Exploring the artistry behind cinema, TV, and ...",None,"[https://en.wikipedia.org/wiki/Entertainment, ...","movies film ""video essays"" ""film essay""",UUUyvQV2JsICeLZP4c_h40kA,"[{'title': 'Why Some Movies Feel More Alive', ..."


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 251 entries, 0 to 250
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   channel_id    251 non-null    object
 1   channel_name  251 non-null    object
 2   title         251 non-null    object
 3   description   251 non-null    object
 4   country       198 non-null    object
 5   topics        247 non-null    object
 6   keywords      159 non-null    object
 7   uploads       251 non-null    object
 8   videos        251 non-null    object
dtypes: object(9)
memory usage: 17.8+ KB


None

## cleaning

In [19]:
df = df.drop(["title", "uploads"], axis=1)
df = df.reset_index(drop=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 251 entries, 0 to 250
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   channel_id    251 non-null    object
 1   channel_name  251 non-null    object
 2   description   251 non-null    object
 3   country       198 non-null    object
 4   topics        247 non-null    object
 5   keywords      159 non-null    object
 6   videos        251 non-null    object
dtypes: object(7)
memory usage: 13.9+ KB


extracting topic names from wikipedia link

In [20]:
df["topics"] = df["topics"].apply(
    lambda cell: ", ".join(
        url.split("/")[-1].replace("_", " ") for url in cell
    ) if cell is not None else None
)

imputing Nones in topics and keywords with value from other column

In [21]:
df[df["topics"].isna() & df["keywords"].isna()]

,channel_id,channel_name,description,country,topics,keywords,videos
139,UCa840NC-ZxtMIZ5wYnBcn7w,Doug Woolever,,None,None,None,[{'title': 'Spongebob Squarepants: Skin Theory...


for this one i'll manually fill in topics and keywords. for others ill use one to fill the other

In [23]:
df.at[139, "country"] = "US"
df.at[139, "topics"] = "Entertainment"
df.at[139, "keywords"] = "TV analysis theory spongebob"

In [24]:
df.iloc[139]

channel_id                               UCa840NC-ZxtMIZ5wYnBcn7w
channel_name                                        Doug Woolever
description                                                      
country                                                        US
topics                                              Entertainment
keywords                             TV analysis theory spongebob
videos          [{'title': 'Spongebob Squarepants: Skin Theory...
Name: 139, dtype: object

In [28]:
df["keywords"] = df["keywords"].str.replace("\"", "", regex=False)
df["topics"] = df["topics"].fillna(df["keywords"])
df["keywords"] = df["keywords"].fillna(df["topics"])
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 251 entries, 0 to 250
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   channel_id    251 non-null    object
 1   channel_name  251 non-null    object
 2   description   251 non-null    object
 3   country       199 non-null    object
 4   topics        251 non-null    object
 5   keywords      251 non-null    object
 6   videos        251 non-null    object
dtypes: object(7)
memory usage: 13.9+ KB


removing description from video column dictionary

In [36]:
df["videos"] = df["videos"].apply(lambda row: " ".join([v["title"] for v in row ]))

In [37]:
df.head()

,channel_id,channel_name,description,country,topics,keywords,videos
0,UCqgh7QNlVV8kv3LBTJZYWrg,The Film Essay,A video essay channel devoted to making videos...,GB,"Television program, Film, Entertainment",film essay video essay film analysis film revi...,Why Christopher Nolan uses time as architectur...
1,UCjFqcJQXGZ6T6sxyFB-5i6A,Every Frame a Painting,Taylor & Tony are gone again! Thanks for watch...,None,"Lifestyle (sociology), Entertainment, Film",filmmaking film analysis video essay,Every Frame x Lackadaisy - Interview with Trac...
2,UCr8r7UVsDaRiIQSVUHVBd_A,Nikki Carreon,"Hello!! Just here making silly stuff, making s...",US,"Lifestyle (sociology), Entertainment, Film","Lifestyle (sociology), Entertainment, Film",Why I Went to a MENTAL HOSPITAL. Stranger Thin...
3,UCjKSoJlPgcK6BmoSqXpj5xQ,Action Button,patreon.com/actionbutton\ntwitch.tv/actionbutt...,None,"Role-playing video game, Action-adventure game...","Role-playing video game, Action-adventure game...","ACTION BUTTON PICTURES PRESENTS ""LOS ANGELES N..."
4,UCUyvQV2JsICeLZP4c_h40kA,Thomas Flight,"Exploring the artistry behind cinema, TV, and ...",None,"Entertainment, Film, Lifestyle (sociology)",movies film video essays film essay,Why Some Movies Feel More Alive Top 10 Films o...
